In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings("ignore")
from sklearn.feature_extraction.text import CountVectorizer , TfidfVectorizer

In [3]:
df = pd.read_csv("cleaned_data.csv")

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      4803 non-null   int64 
 1   title   4803 non-null   object
 2   tags    4803 non-null   object
dtypes: int64(1), object(2)
memory usage: 112.7+ KB


CountVectorizer/TfidfVectorizer is a tool in scikit-learn that converts text data into numbers.

CountVectorizer counts how many times each word appears in each document.
TfidfVectorizer does the same, but it also takes into account how important a word is in the entire dataset. 
It gives less weight to common words that appear in many documents and more weight to words that are unique to a few documents.

When you use CountVectorizer or TfidfVectorizer, you can specify parameters like max_features to limit the number of features (words) to keep. In this case, we want to keep only the top 10,000 most frequent words from the dataset.
Use only the top 10,000 most frequent words from the dataset. i.e (all movies x 10000 unique words) shaped matrix 
Remove common English words that do not add meaning.

When .fit_transform() runs, CountVectorizer calls an internal, built-in python function called a tokenizer. 
By default, it uses a regular expression (r"(?u)\b\w\w+\b") to split your long string of tags into individual words of 2 or more characters.
You didn't have to write code for it because scikit-learn did the heavy lifting automatically behind the scenes.
 
E.g.
lets say vectorizer finds 35,000 unique words.
Instead of keeping all 35,000, it sorts them by their term frequency (how many times they appear across the whole dataset) from highest to lowest. 
It then slices off the top 10,000 most frequent words and throws away the rest.

In [5]:
# # Train without a limit to see the true size of your vocabulary
# temp_cv = CountVectorizer(stop_words='english')
# temp_cv.fit(df['tags'])

# print("Total unique words in your dataset:", len(temp_cv.vocabulary_))

In [6]:
# cv = CountVectorizer(max_features = 10000, stop_words = 'english')
# vector = cv.fit_transform(df['tags']).toarray()       # train cv on 'tags' column
# vector

In [7]:
# ------------------------------------ TF-IDF ------------------------------------

# TF-IDF prioritizes unique plot elements and specific actor/director names over generic tags,
# Your Recommendations will feel much more accurate and "smart" to the end user.

tfidf = TfidfVectorizer(max_features = 10000, stop_words='english')
vector = tfidf.fit_transform(df['tags']).toarray()          

In [8]:
vector.shape

(4803, 10000)

In [11]:
# ------------------------------------ COSINE SIMILARITY ------------------------------------

# Now we will find cosine similarity
# Cosine Similarity measures how similar two texts are by comparing the angle between their vectors(not a magnitude)
# cosine similarity compares every document with every other document i.e in our case similarity will be of (4803,4803) shape.

# formula 

#                                                      A . B
#                 cosine_similarity(A, B) = ---------------------------
#                                                 ||A|| . ||B||

# where,
#     A = vector representation of document A
#     B = vector representation of document B
#     ||A|| = magnitude of vector A
#     ||B|| = magnitude of vector B

# Value ranges from 0 to 1
# 1  => exactly similar
# 0  => no similarity
# -1 => opposite (rare in text based systems)


# Key Advantages:
#         Ignores magnitude => focuses on pattern
#         Works great with:
#             text (TF-IDF)
#             embeddings (BERT, SentenceTransformers)
#         Fast & scalable
#         Works well for high-dimensional sparse data

# | Step                | Shape         | Meaning       |
# | ------------------- | ------------- | ------------- |
# | Vectorized Data (X) | (4803, 10000) | docs vs words |
# | Cosine Similarity   | (4803, 4803)  | docs vs docs  |


Vectors:

A = [1, 2, 3]
B = [2, 4, 6]
Step 1: Dot Product

A . B = (1 x 2) + (2 x 4) + (3 x 6) = 2 + 8 + 18 = 28

Step 2: Magnitude

||A|| = \sqrt{1^2 + 2^2 + 3^2} = \sqrt{14}

||B|| = \sqrt{2^2 + 4^2 + 6^2} = \sqrt{56}

Step 3: Final Similarity

                   28                     28            28
     =  ------------------------- = -------------- = -------- = 1
            \sqrt(14).\sqrt(56)       \sqrt(784)        28

Cosine similarity = 1
=> vectors are perfectly aligned (same direction)
=> highly similar

When a single matrix is passed to cosine_similarity, it computes pairwise cosine similarity between all rows, returning a similarity matrix.

In [12]:
similarity = cosine_similarity(vector)
similarity

array([[1.        , 0.08995431, 0.13527465, ..., 0.03515908, 0.02038775,
        0.02899445],
       [0.08995431, 1.        , 0.05279691, ..., 0.01930291, 0.01193047,
        0.01502296],
       [0.13527465, 0.05279691, 1.        , ..., 0.03194226, 0.03031955,
        0.00827667],
       ...,
       [0.03515908, 0.01930291, 0.03194226, ..., 1.        , 0.03032097,
        0.02804812],
       [0.02038775, 0.01193047, 0.03031955, ..., 0.03032097, 1.        ,
        0.01366794],
       [0.02899445, 0.01502296, 0.00827667, ..., 0.02804812, 0.01366794,
        1.        ]], shape=(4803, 4803))

In [13]:

similarity.shape


(4803, 4803)

In [14]:
#Lets write function get name of movie by index
def get_name_by_index(i):
    if i < len(df) and i>0:
        return df.loc[i,'title']
    else:
        return''

In [15]:
a = get_name_by_index(10)
a

'Superman Returns'

In [16]:
# # Function to improvise the logic for getting the name.
# # if user enter the spider man it will give error as in real dataset name is like spider-man.
# # so it will not match the movie name. it will show movie not found. so we need to handle spaces and -  in this case.

# def get_index_from_name(name):
#     # Normalize user input: lowercase it and strip out all spaces and hyphens
#     clean_user_name = name.strip().lower().replace(' ', '').replace('-', '')
    
#     # Vectorized pandas match: normalize the dataframe column for comparison
#     match = df[df['title'].str.lower().str.replace(' ', '').str.replace('-', '') == clean_user_name]
    
#     if not match.empty:
#         return match.index[0]
#     return -1

# #Lets write function get index of movie by movie name

# def get_index_from_name(name):
#     found_index = -1
#     for i in df.index:
#         if df.loc[i,'title'].strip().lower() == name.strip().lower():
#             found_index = i
#             break
#     return found_index

In [17]:
# b = get_index_from_name("Spider-man 3")
# b

In [18]:
# name = input("Enter movie you watched:")
# index = get_index_from_name(name)
# if index != -1:
#     similarity_indexes =  list(enumerate(similarity[index]))
#     similarity_indexes = sorted(similarity_indexes, key=lambda x: x[1], reverse=True)
#     print("You must watch following movies:")
#     for i in range(1, 6):
#         print(i, ":", get_name_by_index(similarity_indexes[i][0]))
# else:
#     print("Movie Not Found")    
    
    

In [19]:
# Function to improvise the logic for getting the name.
# if user enter the spider man it will give error as in real dataset name is like spider-man.
# so it will not match the movie name. it will show movie not found. so we need to handle spaces and -  in this case.

def get_index_from_name(name):
    # Normalize user input: lowercase it and strip out all spaces and hyphens
    clean_user_name = name.strip().lower().replace(' ', '').replace('-', '')
    
    # Vectorized pandas match: normalize the dataframe column for comparison
    match = df[df['title'].str.lower().str.replace(' ', '').str.replace('-', '') == clean_user_name]
    # spider man 3 == spider man 3
    if not match.empty:
        return match.index[0]
    return -1

In [20]:
get_index_from_name("spider man 3")

np.int64(5)

In [21]:
name = input("Enter movie you watched:")
index = get_index_from_name(name)
# similarity_indexes = list(enumerate(similarity[index]))
# similarity_indexes = sorted(similarity_indexes, key=lambda x: x[1], reverse=True)
# similarity_indexes

if index != -1:
    # Fetch similarity rankings. enumerate() pairs each score with its original movie index like (5, np.float64(0.3635931236853142))
    similarity_indexes = list(enumerate(similarity[index]))

    # The very first item in this sorted list (index 0) will always be the movie itself with a perfect similarity score of 1.0
    similarity_indexes = sorted(similarity_indexes, key=lambda x: x[1], reverse=True)
    
    print(f"\nBecause you watched '{df.loc[index, 'title']}':")
    print("You must watch the following movies:")
    
    # Loop to print top 5 recommendations (skipping index 0 as it's the movie itself)
    for i in range(1, 6):
        movie_idx = similarity_indexes[i][0]
        print(i, ":", get_name_by_index(movie_idx))
else:
    print("Movie Not Found")

Enter movie you watched: intersteller


Movie Not Found


In [22]:
#Lets export similarity 
import pickle as pkl
pkl.dump(similarity, open('similarity.pkl', 'wb'))